In [ ]:
# this script should take baseball side angle monocular footage, perform video segmentation of the athlete as he moves
# prompted by manual clicking (see below)

# script then colour codes the segmentation as magma, so the denser centers of the segmentation have more weight than noisy edges
# finally, perform isomap, a statistical technique that converts high dimensional data into 2d manifolds

# ideas originate from "Human Motion Recognition Using Isomap and Dynamic Time Warping" by Blackburn and Ribeiro

# this colab script originates as the download_and_click.py file on my github
# the reason there are 2 separate scripts is because Colab cannot pop-up a separate window for manual click prompting
# also unsure if I could run yt-dlp from Colab
# then once I have the original zip file from that script, take it here to access free GPU usage in colab

# my bluesky account where you can see updates on my baseball work: https://bsky.app/profile/ryangunther1.bsky.social

In [ ]:
# set up for GPU
import os, sys, re, glob, json, shutil, zipfile, cv2, numpy as np
from pathlib import Path
from IPython.display import HTML, display
from google.colab import files

def show_video(path, width=768):
    """inline HTML5 vid player"""
    path = str(path)
    return display(HTML(f"""
    <video width="{width}" controls>
      <source src="{path}" type="video/mp4">
      Your browser does not support the video tag.
    </video>
    """.strip()))

In [ ]:
# run this only first time per runtime (it will take a while because of installs and uploads)

existing_zips = sorted([f for f in os.listdir("/content") if f.endswith("_upload_ready.zip")])
existing_pts  = sorted([f for f in os.listdir("/content") if f.endswith(".pt")])

if existing_zips:
    zip_name = existing_zips[-1]
    print(f"found existing upload_ready zip: {zip_name}")
else:
    print("upload *_upload_ready.zip")
    up1 = files.upload()
    zip_name = next(iter(up1.keys()))
    assert zip_name.endswith(".zip")

if existing_pts:
    ckpt_name = existing_pts[-1]
    print(f"found existing SAM2 checkpoint: {ckpt_name}")
else:
    print("upload SAM2 checkpoint .pt")
    up2 = files.upload()
    ckpt_name = next(iter(up2.keys()))
    assert ckpt_name.endswith(".pt")

# auto clear old folders
for d in ["masks_bw", "masks_dist_gray", "masks_dist_magma",
          "magma_overlays", "sam2/frames_jpg_fixed"]:
    full = f"/content/{d}"
    if os.path.exists(full):
        shutil.rmtree(full, ignore_errors=True)

# unzip and detect assets
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content")

# detect fixed video, manual points, contact sheet from VS code script
fixed_mp4 = None
manual_json = None
contact_sheet = None

for f in os.listdir("/content"):
    if f.endswith("_fixed.mp4"):
        fixed_mp4 = f"/content/{f}"
    elif f.startswith("manual_points_") and f.endswith(".json"):
        manual_json = f"/content/{f}"
    elif f == "contact_sheet.jpg":
        contact_sheet = f"/content/{f}"

assert fixed_mp4 and manual_json, "missing mp4 or json"

# derive prefix from fixed_mp4 filename
m = re.match(r"(.+)_fixed\.mp4$", Path(fixed_mp4).name)
assert m, "could not parse"
PREFIX = m.group(1)
print("Detected PREFIX:", PREFIX)

# install SAM2 (clone once if not present)
if not os.path.exists("/content/sam2_repo_installed"):
    if not os.path.exists("/content/sam2"):
        !git clone https://github.com/facebookresearch/sam2.git
    %cd /content/sam2
    !pip install -e .
    !pip install -e ".[notebooks]" > /dev/null
    Path("/content/sam2_repo_installed").touch()   # marker file
else:
    %cd /content/sam2
    print("SAM2 already installed - skipping")

In [ ]:
import torch, gc, shutil, os

# free memory
try:
    del predictor
    del state
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# delete existing folders (ignore if they don't exist)
dirs_to_clear = [
    "/content/sam2/frames_jpg_fixed",
    "/content/masks_bw",
    "/content/sam2/hitter_overlays_bbox",
    "/content/sam2/hitter_masks_bw",
    "/content/sam2/hitter_overlays",
    "/content/magma_overlays",
    "/content/masks_dist_gray",
    "/content/masks_dist_magma"
]

for d in dirs_to_clear:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

print("old frames/masks/overlays deleted, ready for new clip")

In [ ]:
!mv /content/sam2 /content/sam2_repo

In [ ]:
# only once per runtime

import sys
sys.path.insert(0, "/content/sam2_repo")

In [ ]:
%cd /content/sam2_repo
!pip install -e .
!pip install -e ".[notebooks]"

Path("/content/sam2_repo_installed").touch()

In [ ]:
import os, cv2, json, shutil
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2_video_predictor
from collections import defaultdict
from matplotlib import pyplot as plt
from pathlib import Path

# paths and config
VIDEO_PATH = fixed_mp4
CHECKPOINT = f"/content/{ckpt_name}"
MODEL_CFG  = "configs/sam2.1/sam2.1_hiera_b+.yaml"
FRAMES_DIR = "/content/sam2/frames_jpg_fixed"

# clear old frames and rebuild dir
shutil.rmtree(FRAMES_DIR, ignore_errors=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

# read frames from video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("could not open video, double check mp4 type")

fps = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame)
cap.release()
print(f"Loaded {len(frames)} frames at {fps:.2f} fps")

# save frames to disk for SAM2
for i, f in enumerate(frames):
    Image.fromarray(f).save(os.path.join(FRAMES_DIR, f"{i:04d}.jpg"), quality=95)
print(f"saved frames to {FRAMES_DIR}")

# init predictor and state
predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT)
state = predictor.init_state(FRAMES_DIR)
print("SAM2 initialized successfully")

# load manual click json
with open(manual_json, "r") as f:
    manual_points = json.load(f)
print(f"loaded manual prompts for {len(manual_points)} frames")

# group clicks by frame
frame_prompts = defaultdict(lambda: {"points": [], "labels": []})
for frame_key, pts in manual_points.items():
    frame_idx = int(frame_key)
    pos = np.array(pts.get("pos", []), dtype=float)
    neg = np.array(pts.get("neg", []), dtype=float)

    # comment out these two lines if you want to force SAM2 to process every frame
    if len(pos) + len(neg) == 0:
        continue

    if pos.size:
        frame_prompts[frame_idx]["points"].extend(pos.tolist())
        frame_prompts[frame_idx]["labels"].extend([1] * len(pos))
    if neg.size:
        frame_prompts[frame_idx]["points"].extend(neg.tolist())
        frame_prompts[frame_idx]["labels"].extend([0] * len(neg))

# seed SAM2 from manual clicking
obj_id = 0
for frame_idx in sorted(frame_prompts.keys()):
    pts_this_frame = np.array(frame_prompts[frame_idx]["points"], dtype=float)
    labels_this_frame = np.array(frame_prompts[frame_idx]["labels"], dtype=int)
    if pts_this_frame.size == 0:
        continue

    _fi, _obj_ids, _masks = predictor.add_new_points_or_box(
        inference_state=state,
        frame_idx=frame_idx,
        obj_id=obj_id,
        points=pts_this_frame,
        labels=labels_this_frame
    )
    print(f"seeded frame {frame_idx} with {len(labels_this_frame)} clicks")

# propagate segmentation through the vid
masks_bw_dir = "/content/masks_bw"
os.makedirs(masks_bw_dir, exist_ok=True)

all_masks = []
all_bboxes = []

for frame_idx, object_ids, masks in predictor.propagate_in_video(state):
    # 1: get raw mask as float; DO NOT cast to uint8 yet
    mask_raw = masks[0].detach().cpu().numpy().squeeze()

    # 2: ensure it's a probability in [0,1]; if logits, apply sigmoid
    if mask_raw.max() > 1.0 or mask_raw.min() < 0.0:
        mask_prob = 1.0 / (1.0 + np.exp(-mask_raw))
    else:
        mask_prob = mask_raw.astype(np.float32)

    # 3: binarize (no premature uint8 casting elsewhere)
    mask_bin = (mask_prob > 0.5).astype(np.uint8) * 255

    # optional light cleanup that won’t bridge holes
    kernel = np.ones((3, 3), np.uint8)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN,  kernel, iterations=1)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel, iterations=1)

    # 4: redraw using hierarchy so OUTER is white and INNER (holes) are black
    contours, hierarchy = cv2.findContours(mask_bin, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(mask_bin)
    if hierarchy is not None:
        for i, h in enumerate(hierarchy[0]):
            if h[3] == -1:   # outer contour (no parent)
                cv2.drawContours(filled, contours, i, 255, thickness=cv2.FILLED)
            else:            # inner contour (hole)
                cv2.drawContours(filled, contours, i, 0,   thickness=cv2.FILLED)
    mask_bin = filled

    # bbox on the filled mask
    ys, xs = np.where(mask_bin > 0)
    if len(xs) and len(ys):
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        bbox = [int(x_min), int(y_min), int(x_max), int(y_max)]
    else:
        bbox = None

    cv2.imwrite(f"{masks_bw_dir}/{frame_idx:04d}.png", mask_bin)
    all_masks.append(mask_bin)
    all_bboxes.append(bbox)

print(f"propagation complete. {len(all_masks)} binary masks saved in {masks_bw_dir}")

# 9: quick sanity check (few sample masks)
sample_idxs = list(range(0, len(all_masks), max(1, len(all_masks)//6)))[:6]
plt.figure(figsize=(12, 6))
for i, idx in enumerate(sample_idxs, 1):
    plt.subplot(2, 3, i)
    plt.imshow(all_masks[idx], cmap="gray")
    plt.title(f"Mask {idx}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# distance transform from masks_bw to grayscale distance images
dist_gray_dir = "/content/masks_dist_gray"
dist_magma_dir = "/content/masks_dist_magma"
os.makedirs(dist_gray_dir, exist_ok=True)
os.makedirs(dist_magma_dir, exist_ok=True)

mask_paths = sorted(glob.glob(f"{masks_bw_dir}/*.png"))
assert len(mask_paths) > 0, "No masks found in /content/masks_bw."

for p in mask_paths:
    fname = Path(p).name
    mask = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # dist transform from foreground pixels to nearest edge
    dist = cv2.distanceTransform(mask_bin, cv2.DIST_L2, 0)

    # gentle nonlinearity
    dist_norm = np.power(dist, 0.5)

    # normalize to 0..255 uint8 for storage
    dist_vis = cv2.normalize(dist_norm, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # save grayscale distance for isomap
    cv2.imwrite(str(Path(dist_gray_dir)/fname), dist_vis)

    # save magma-colored preview
    colored = cv2.applyColorMap(dist_vis, cv2.COLORMAP_MAGMA)
    cv2.imwrite(str(Path(dist_magma_dir)/fname.replace(".png", "_magma.png")), colored)

print("distance maps created: grayscale for Isomap, + magma previews for visualization")

# create magma overlay video on original background with bbox
magma_overlay_dir = "/content/magma_overlays"
os.makedirs(magma_overlay_dir, exist_ok=True)

# we will need the original frames again
# if memory concerns, re-read from disk
# frames already in 'frames' from Part 1, reuse
for i, p in enumerate(mask_paths):
    gray_path = Path(dist_gray_dir)/Path(p).name
    gray = cv2.imread(str(gray_path), cv2.IMREAD_GRAYSCALE)
    color_magma = cv2.applyColorMap(gray, cv2.COLORMAP_MAGMA)

    frame_rgb = frames[i].copy()
    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

    # composite: only where mask > 0, place magma color; otherwise keep original
    mask_bin = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    mask_fg = (mask_bin > 0)

    # mostly stylistic but:
    # blend magma on foreground only. use full replace or alpha blend if desired
    # here, full replace on masked pixels:
    frame_bgr[mask_fg] = color_magma[mask_fg]

    # draw bbox
    bbox = all_bboxes[i]
    if bbox is not None:
        x0, y0, x1, y1 = bbox
        cv2.rectangle(frame_bgr, (x0, y0), (x1, y1), (0, 255, 0), 2)

    cv2.imwrite(f"{magma_overlay_dir}/{i:04d}.jpg", frame_bgr)

# render to mp4 with source fps
magma_video = "/content/magma_overlay.mp4"
!ffmpeg -y -framerate {fps} -i {magma_overlay_dir}/%04d.jpg -c:v libx264 -pix_fmt yuv420p {magma_video}

print("magma overlay video saved:", magma_video)
show_video(magma_video, width=768)

In [ ]:
# install Isomap dependencies
!pip -q install scikit-learn matplotlib

import os, cv2, glob, numpy as np
from sklearn.manifold import Isomap
import matplotlib.pyplot as plt

# set the dir of grayscale distance masks
dist_gray_dir = "/content/masks_dist_gray"

# load grayscale images as flat vectors
dist_gray_paths = sorted(glob.glob(f"{dist_gray_dir}/*.png"))
frames_data = []
for fp in dist_gray_paths:
    img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
    frames_data.append(img.flatten())

Y = np.array(frames_data, dtype=np.float32)
n_frames = Y.shape[0]
print("Matrix shape:", Y.shape)

n_neighbors = max(2, int(round(n_frames / 3)))
if n_neighbors >= n_frames:
    raise ValueError(f"n_neighbors={n_neighbors} must be < n_frames={n_frames}")

iso = Isomap(n_neighbors=n_neighbors, n_components=2)
X_iso = iso.fit_transform(Y)

# plot + save
plt.figure(figsize=(8, 6))
plt.plot(X_iso[:, 0], X_iso[:, 1], "-o", markersize=3)
plt.title(f"isomap of swing (neighbors={n_neighbors})")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.tight_layout()

iso_plot_path = f"/content/{PREFIX}_isomap_plot.png"
plt.savefig(iso_plot_path, dpi=150)
plt.show()

In [ ]:
# SAVE ISOMAP RESULTS + MAGMA VIDEO + NOTES → ZIP

import os, glob, zipfile, shutil
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
from IPython.display import display
import ipywidgets as widgets

# locate upload folder automatically
upload_zips = sorted(glob.glob("/content/*_upload_ready.zip"))
if not upload_zips:
    raise FileNotFoundError("no *_upload_ready.zip found in /content/")
upload_zip_path = upload_zips[-1]
upload_root = os.path.splitext(upload_zip_path)[0]
os.makedirs(upload_root, exist_ok=True)

print(f"using upload folder:\n{upload_root}")

# save Isomap plot and CSV directly into that folder
iso_plot_path = os.path.join(upload_root, f"{PREFIX}_isomap_plot.png")
iso_csv_path  = os.path.join(upload_root, f"{PREFIX}_isomap_coords.csv")

# plot isomap embedding
plt.figure(figsize=(8, 6))
plt.plot(X_iso[:, 0], X_iso[:, 1], "-o", markersize=3)
plt.title(f"Isomap of swing (neighbors={max(2, int(round(n_frames/3)))})")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.tight_layout()
plt.savefig(iso_plot_path, dpi=150)
plt.close()

# save CSV
df = pd.DataFrame({"frame": np.arange(n_frames), "comp1": X_iso[:, 0], "comp2": X_iso[:, 1]})
df.to_csv(iso_csv_path, index=False)

print(f"saved Isomap plot in {iso_plot_path}")
print(f"saved Isomap CSV  in {iso_csv_path}")

# move magma overlay there too
magma_src = "/content/magma_overlay.mp4"
magma_dst = os.path.join(upload_root, "magma_overlay.mp4")
if os.path.exists(magma_src):
    shutil.move(magma_src, magma_dst)
    print(f"moved magma overlay to {magma_dst}")
else:
    print("magma_overlay.mp4 not found")

# notes input box
notes_box = widgets.Textarea(
    placeholder="type notes here. If no prompts before the first 3 frames just say standard minimal prompting",
    description="Notes:",
    layout=widgets.Layout(width="100%", height="100px")
)
display(notes_box)

def save_notes(_):
    notes_path = os.path.join(upload_root, "prompt_notes.txt")
    with open(notes_path, "w") as f:
        f.write(notes_box.value.strip() + "\n")
    print(f"notes saved in {notes_path}")

save_button = widgets.Button(description="Save Notes", button_style="success")
save_button.on_click(save_notes)
display(save_button)

In [ ]:
# RE-ZIP FOLDER WITH ISOMAP OUTPUTS + NOTES

with zipfile.ZipFile(upload_zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for root, _, files_ in os.walk(upload_root):
        for f in files_:
            abs_path = os.path.join(root, f)
            rel_path = os.path.relpath(abs_path, start=os.path.dirname(upload_root))
            z.write(abs_path, arcname=rel_path)

print(f"updated zip: {upload_zip_path}")
files.download(upload_zip_path)

In [ ]:
template_csv = "/content/ErnieClement_BlueJays_2025-08-10_00-11_isomap_coords.csv"

template_df = pd.read_csv(template_csv)
template = template_df[["comp1", "comp2"]].values

In [ ]:
# INTERPOLATE TO FIXED LENGTH + DTW DISTANCE (2D) -> 1D
# not really using the template portion of this too much but keeping it for future possibilities

!pip -q install dtaidistance

import pandas as pd
import numpy as np
from dtaidistance import dtw

# load current swing
coords_test = pd.read_csv("/content/RonaldAcunaJr_Braves_2025-07-18_00-02_upload_ready/RonaldAcunaJr_Braves_2025-07-18_00-02_fixed_isomap_coords.csv")
traj_test = coords_test[["comp1", "comp2"]].values  # Nx2

# load template swing (Ernie)
coords_temp = pd.read_csv(template_csv)
traj_temp = coords_temp[["comp1", "comp2"]].values

# interpolation function
def interp_manifold(arr, new_len=100):
    old_len = arr.shape[0]
    old_t = np.linspace(0, 1, old_len)
    new_t = np.linspace(0, 1, new_len)

    new_arr = np.vstack([
        np.interp(new_t, old_t, arr[:,0]),
        np.interp(new_t, old_t, arr[:,1])
    ]).T
    return new_arr

# 2D to 1D scalar sequence (eucl magnitude)
def to_scalar(arr):
    return np.sqrt(arr[:,0]**2 + arr[:,1]**2)

# interpolate both first
temp_interp = interp_manifold(traj_temp,  new_len=100)
test_interp = interp_manifold(traj_test, new_len=100)

# convert to 1D sequences for DTW
seq_temp = to_scalar(temp_interp)
seq_test = to_scalar(test_interp)

# dtw distance
dist = dtw.distance(seq_test, seq_temp)
print(f"DTW distance to template: {dist:.4f}")

# plot interpolated manifolds for visual check
plt.figure(figsize=(6,6))
plt.plot(temp_interp[:,0], temp_interp[:,1], 'o-', label="Template (Ernie)")
plt.plot(test_interp[:,0], test_interp[:,1], 'o-', label="Test swing")
plt.title("Interpolated Manifolds (100 frames)")
plt.legend()
plt.xlabel("comp1")
plt.ylabel("comp2")
plt.show()